<a href="https://colab.research.google.com/github/v35001/aoid-team2-ahiyp/blob/main/code/11_regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# ==============================================================================
# 1. 패키지 설치 및 구글 드라이브 연결, 원자료 불러오기
# ==============================================================================

# 필요한 패키지 먼저 설치
!pip -q install pyreadstat

from google.colab import drive
drive.mount('/content/drive')

import os
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
import pyreadstat
from IPython.display import display

# 데이터가 들어 있는 폴더 (구글 드라이브 루트 폴더로 수정)
DATA_DIR = '/content/drive/MyDrive'

# 파일 위치 자동 검색
file_path = os.path.join(DATA_DIR, 'hn24_all.sas7bdat')

if not os.path.exists(file_path):
    for root, dirs, files in os.walk(DATA_DIR):
        if 'hn24_all.sas7bdat' in files:
            file_path = os.path.join(root, 'hn24_all.sas7bdat')
            break

if not os.path.exists(file_path):
    raise FileNotFoundError(
        'hn24_all.sas7bdat 파일을 구글 드라이브에서 찾을 수 없습니다.'
    )

cols = [
    'age', 'sex', 'HE_DM_HbA1c', 'HE_HP', 'HE_obe',
    'incm', 'edu', 'BS3_1', 'BD1_11',
    'wt_itvex', 'kstrata', 'psu'
]

df, meta = pyreadstat.read_sas7bdat(
    file_path,
    usecols=cols
)

df.columns = df.columns.str.lower()

print('원자료 불러오기 완료')
print('파일 위치:', file_path)
print('원자료 크기:', df.shape)


# ==============================================================================
# 2. 변수 정의 및 전처리
# ==============================================================================

# 당뇨병: 0=비유병, 1=유병
df['dm'] = np.where(
    df['he_dm_hba1c'].isin([1, 2]), 0,
    np.where(df['he_dm_hba1c'] == 3, 1, np.nan)
)

# 고혈압: 0=비유병, 1=유병
df['hp'] = np.where(
    df['he_hp'].isin([1, 2, 3]), 0,
    np.where(df['he_hp'] == 4, 1, np.nan)
)

# 성별: 0=여자, 1=남자
df['sex_cat'] = np.where(
    df['sex'] == 2, 0,
    np.where(df['sex'] == 1, 1, np.nan)
)

# 연령: 만 19세 이상
df['age_num'] = np.where(
    df['age'] >= 19, df['age'], np.nan
)

# 비만: 0=비만 아님, 1=비만
df['obe_cat'] = np.where(
    df['he_obe'].isin([1, 2, 3]), 0,
    np.where(df['he_obe'].isin([4, 5, 6]), 1, np.nan)
)

# 소득수준
df['incm_cat'] = df['incm'].where(
    df['incm'].isin([1, 2, 3, 4]), np.nan
)

# 교육수준
df['edu_cat'] = df['edu'].where(
    df['edu'].isin([1, 2, 3, 4]), np.nan
)

# 흡연: 0=비흡연, 1=현재 흡연
df['smk_cat'] = np.where(
    df['bs3_1'].isin([3, 8]), 0,
    np.where(df['bs3_1'].isin([1, 2]), 1, np.nan)
)

# 음주: 0=월 1회 미만/안마심, 1=월 1회 이상
df['drk_cat'] = np.where(
    df['bd1_11'].isin([1, 2, 8]), 0,
    np.where(df['bd1_11'].isin([3, 4, 5, 6]), 1, np.nan)
)



# ==============================================================================
# C-2. 복합표본 로지스틱 회귀분석 및 Table 2 최종 출력
# ==============================================================================

import os
import tempfile
import subprocess
import numpy as np
import pandas as pd
from IPython.display import display

# ==============================================================================
# 1. 변수 및 자료 확인
# ==============================================================================

required_vars = [
    'age', 'dm', 'hp', 'age_num', 'sex_cat',
    'incm_cat', 'edu_cat', 'smk_cat', 'drk_cat',
    'obe_cat', 'wt_itvex', 'kstrata', 'psu'
]

missing_vars = [v for v in required_vars if v not in df.columns]

if missing_vars:
    raise ValueError(f'분석에 필요한 변수가 없습니다: {missing_vars}')

print('=' * 70)
print('[1. 변수별 범주화 및 결측값 확인]')
print('=' * 70)

check_vars = [
    'dm', 'hp', 'sex_cat', 'obe_cat',
    'incm_cat', 'edu_cat', 'smk_cat', 'drk_cat'
]

for var in check_vars:
    print(f'\n[{var}]')
    print(df[var].value_counts(dropna=False).sort_index())


# ==============================================================================
# 2. R 복합표본 분석 환경 준비
# ==============================================================================

!apt-get -qq install -y r-base

!Rscript -e "if (!requireNamespace('survey', quietly=TRUE)) install.packages('survey', repos='https://cloud.r-project.org')"

temp_dir = tempfile.mkdtemp()

input_csv = os.path.join(temp_dir, 'analysis_data.csv')
output_csv = os.path.join(temp_dir, 'survey_result.csv')
r_script = os.path.join(temp_dir, 'survey_analysis.R')

# 원자료와 범주화 변수를 R로 전달
df.to_csv(input_csv, index=False)


# ==============================================================================
# 3. R 복합표본 설계 및 로지스틱 회귀분석
# ==============================================================================

r_code = r'''
library(survey)

options(survey.lonely.psu = "adjust")

args <- commandArgs(trailingOnly=TRUE)

dat <- read.csv(
  args[1],
  stringsAsFactors=FALSE,
  na.strings=c("", "NA")
)

# 복합표본 설계변수 확인
design_vars <- c("wt_itvex", "kstrata", "psu")

if (!all(design_vars %in% names(dat))) {
  stop("복합표본 설계변수가 누락되었습니다.")
}

# 가중치, 층화, 집락 정보가 유효한 표본 확인
dat <- dat[
  complete.cases(dat[, design_vars]) &
  is.finite(dat$wt_itvex) &
  dat$wt_itvex > 0,
]

# 노출변수 기준 범주 설정
# 0 = 당뇨병 비유병
# 1 = 당뇨병 유병
dat$dm_f <- factor(
  dat$dm,
  levels=c(0, 1)
)

# 전체 표본을 이용한 복합표본 설계 설정
design_all <- svydesign(
  ids=~psu,
  strata=~kstrata,
  weights=~wt_itvex,
  data=dat,
  nest=TRUE
)

# 만 19세 이상이며 노출 및 결과변수가 확인되는 대상자
# 전체 설계를 먼저 설정한 후 도메인 분석
design_t1 <- subset(
  design_all,
  age >= 19 &
  !is.na(dm) &
  !is.na(hp) &
  dm %in% c(0, 1) &
  hp %in% c(0, 1)
)

# 모형 1: 무보정 로지스틱 회귀분석
design_m1 <- design_t1

model1 <- svyglm(
  hp ~ dm_f,
  design=design_m1,
  family=quasibinomial()
)

# 모형 2: DAG 기반 7개 변수 보정
design_m2 <- subset(
  design_t1,
  !is.na(age_num) &
  !is.na(sex_cat) &
  !is.na(incm_cat) &
  !is.na(edu_cat) &
  !is.na(smk_cat) &
  !is.na(drk_cat) &
  !is.na(obe_cat)
)

model2 <- svyglm(
  hp ~ dm_f +
    age_num +
    factor(sex_cat) +
    factor(incm_cat) +
    factor(edu_cat) +
    factor(smk_cat) +
    factor(drk_cat) +
    factor(obe_cat),
  design=design_m2,
  family=quasibinomial()
)

# 회귀분석 결과 추출
get_result <- function(model, design, model_name) {

  term <- "dm_f1"

  coef_table <- coef(summary(model))

  if (!(term %in% names(coef(model)))) {
    stop("당뇨병 노출변수의 회귀계수를 찾을 수 없습니다.")
  }

  beta <- coef(model)[term]

  # 95% 신뢰구간
  ci <- confint(
    model,
    parm=term,
    level=0.95
  )

  # p값
  p_col <- grep(
    "^Pr\\(",
    colnames(coef_table),
    value=TRUE
  )[1]

  p_value <- coef_table[term, p_col]

  data.frame(
    model=model_name,
    OR=exp(beta),
    lower=exp(ci[1, 1]),
    upper=exp(ci[1, 2]),
    p=p_value,
    N=nrow(design$variables)
  )
}

# 모형별 결과
result1 <- get_result(
  model1, design_m1, "모형 1 (무보정)"
)

result2 <- get_result(
  model2, design_m2, "모형 2 (보정)"
)

# R에서 실제 복합표본 분석에 사용된 인원
table1_result <- data.frame(
  model="Table 1",
  OR=NA,
  lower=NA,
  upper=NA,
  p=NA,
  N=nrow(design_t1$variables)
)

# 결과 통합
final_result <- rbind(
  table1_result,
  result1,
  result2
)

write.csv(
  final_result,
  args[2],
  row.names=FALSE
)

cat("복합표본 로지스틱 회귀분석 완료\n")
cat("Table 1 분석 인원:", nrow(design_t1$variables), "명\n")
cat("모형 1 분석 인원:", nrow(design_m1$variables), "명\n")
cat("모형 2 분석 인원:", nrow(design_m2$variables), "명\n")
'''

with open(r_script, 'w', encoding='utf-8') as f:
    f.write(r_code)

# R 분석 실행
run_result = subprocess.run(
    ['Rscript', r_script, input_csv, output_csv],
    capture_output=True,
    text=True
)

print('\n' + run_result.stdout)

if run_result.returncode != 0:
    print(run_result.stderr)
    raise RuntimeError('복합표본 회귀분석 중 오류가 발생했습니다.')


# ==============================================================================
# 4. R 분석 결과 불러오기
# ==============================================================================

survey_result = pd.read_csv(output_csv)

table1_row = survey_result[
    survey_result['model'] == 'Table 1'
].iloc[0]

result1 = survey_result[
    survey_result['model'] == '모형 1 (무보정)'
].iloc[0]

result2 = survey_result[
    survey_result['model'] == '모형 2 (보정)'
].iloc[0]

# 실제 R 복합표본 설계에서 계산한 분석 인원
table1_n = int(table1_row['N'])
n1 = int(result1['N'])
n2 = int(result2['N'])


# ==============================================================================
# 5. C-2. Table 2 회귀분석 결과
# ==============================================================================

def format_p(p):
    if p < 0.001:
        return '<0.001'
    return f'{p:.4f}'


table2 = pd.DataFrame([
    {
        '노출변수': '당뇨병 비유병 (기준군)',
        '모형 1 OR (95% CI)': '1.00 (Ref.)',
        '모형 1 p값': '-',
        '모형 2 OR (95% CI)': '1.00 (Ref.)',
        '모형 2 p값': '-'
    },
    {
        '노출변수': '당뇨병 유병',
        '모형 1 OR (95% CI)': (
            f"{result1['OR']:.3f} "
            f"({result1['lower']:.3f} - {result1['upper']:.3f})"
        ),
        '모형 1 p값': format_p(result1['p']),
        '모형 2 OR (95% CI)': (
            f"{result2['OR']:.3f} "
            f"({result2['lower']:.3f} - {result2['upper']:.3f})"
        ),
        '모형 2 p값': format_p(result2['p'])
    }
])

print('\n' + '=' * 75)
print('[1. C-2. 복합표본 로지스틱 회귀분석 결과]')
print('=' * 75)

display(table2)


# ==============================================================================
# 6. Table 1과 모형별 분석 인원 비교
# ==============================================================================

count_result = pd.DataFrame({
    '구분': ['Table 1', '모형 1', '모형 2'],
    '분석 인원': [
        table1_n,
        n1,
        n2
    ],
    'Table 1과의 차이': [
        0,
        table1_n - n1,
        table1_n - n2
    ]
})

print('\n' + '=' * 75)
print('[2. Table 1과 모형별 분석 인원 비교]')
print('=' * 75)

display(count_result)


# ==============================================================================
# 7. 최종 분석 결과 및 노출 기준 범주 확인
# ==============================================================================

print('\n' + '=' * 75)
print('[3. 최종 분석 결과 확인]')
print('=' * 75)

print(f'Table 1 분석 대상자: {table1_n} 명')
print(f'모형 1 분석 인원: {n1} 명')
print(f'모형 2 분석 인원: {n2} 명')

print(f'모형 1 제외 인원: {table1_n - n1} 명')
print(f'모형 2 제외 인원: {table1_n - n2} 명')

print('\n노출 기준 범주: 당뇨병 비유병 (0)')
print('비교 범주: 당뇨병 유병 (1)')
print('분석 방법: 복합표본 로지스틱 회귀분석')
print('가중치: wt_itvex')
print('층화변수: kstrata')
print('집락변수: psu')

print('\n최종 C-2 Table 2 분석 완료')



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
원자료 불러오기 완료
파일 위치: /content/drive/MyDrive/hn24_all.sas7bdat
원자료 크기: (6997, 12)
[1. 변수별 범주화 및 결측값 확인]

[dm]
dm
0.0    4657
1.0     959
NaN    1381
Name: count, dtype: int64

[hp]
hp
0.0    3816
1.0    2139
NaN    1042
Name: count, dtype: int64

[sex_cat]
sex_cat
0.0    3927
1.0    3070
Name: count, dtype: int64

[obe_cat]
obe_cat
0.0    3748
1.0    2161
NaN    1088
Name: count, dtype: int64

[incm_cat]
incm_cat
1.0    1742
2.0    1739
3.0    1734
4.0    1737
NaN      45
Name: count, dtype: int64

[edu_cat]
edu_cat
1.0    1759
2.0     742
3.0    1930
4.0    2333
NaN     233
Name: count, dtype: int64

[smk_cat]
smk_cat
0.0    5973
1.0     799
NaN     225
Name: count, dtype: int64

[drk_cat]
drk_cat
0.0    3790
1.0    2982
NaN     225
Name: count, dtype: int64

복합표본 로지스틱 회귀분석 완료
Table 1 분석 인원: 5567 명
모형 1 분석 인원: 5567 명
모형 2 분석 인원: 5367 명


[1. C-2. 복합표본 로지스틱 회귀분석

,노출변수,모형 1 OR (95% CI),모형 1 p값,모형 2 OR (95% CI),모형 2 p값
0,당뇨병 비유병 (기준군),1.00 (Ref.),-,1.00 (Ref.),-
1,당뇨병 유병,5.466 (4.583 - 6.520),<0.001,2.356 (1.919 - 2.892),<0.001



[2. Table 1과 모형별 분석 인원 비교]


,구분,분석 인원,Table 1과의 차이
0,Table 1,5567,0
1,모형 1,5567,0
2,모형 2,5367,200



[3. 최종 분석 결과 확인]
Table 1 분석 대상자: 5567 명
모형 1 분석 인원: 5567 명
모형 2 분석 인원: 5367 명
모형 1 제외 인원: 0 명
모형 2 제외 인원: 200 명

노출 기준 범주: 당뇨병 비유병 (0)
비교 범주: 당뇨병 유병 (1)
분석 방법: 복합표본 로지스틱 회귀분석
가중치: wt_itvex
층화변수: kstrata
집락변수: psu

최종 C-2 Table 2 분석 완료
